###import statements

In [ ]:
import os
from pathlib import Path
import pandas as pd
import numpy as np

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
#filePath="C:/users/nyeho/OneDrive/Documents/KzooCS/486/Data/instagram_usage_lifestyle.csv"
filePath = "/content/drive/MyDrive/comp486Final/dataset/instagram_usage_lifestyle.csv"

originData=pd.read_csv(Path(filePath))

###Additional var to hold data to limit computational time wasted in testing

In [ ]:
filth=originData.copy()
label='age'
filth.shape

(1547896, 58)

###Dropping uncessary features

In [ ]:
def getLabels(df, indexes):
    ret=[]
    for num in indexes:
        ret.append(df.columns[num])
    return ret
toDrop=getLabels(filth, [0, 1, 13, 18, 19, 20, 48])
plainData=filth.drop(toDrop, axis=1)
plainData.shape

(1547896, 51)

In [ ]:
#shrinking the instances used: dividing instances used by ten randomly
def shrink(df, ratio=.1) -> pd.DataFrame:
    length=len(df)
    indices=np.random.permutation(length)[:int(ratio*length)]
    return df.iloc[indices].reset_index(drop=True)

#Splitting the numerical and categorical data
def splitDataTypes(df):
    num=[]
    cat=[]
    for column in df.columns:
        if df[column].dtype=='object':
            cat.append(column)
        else:
            num.append(column)
    return num, cat

#splits training and test set
def splitData(df, firstRatio=.9) ->pd.DataFrame:
    length=len(df)
    indices=np.random.permutation(length)
    firstIndices=indices[:int(firstRatio*length)]
    secondIndices=indices[int(firstRatio*length):]
    return df.iloc[firstIndices].reset_index(drop=True), df.iloc[secondIndices].reset_index(drop=True)

#shrinking the data
shrunkData=shrink(plainData)
#assigning numerical and categorical data
nums, cats=splitDataTypes(shrunkData)
#adding labels to temporary var
cats.append(label)
#getting the numerical data columns from shrunken data to be added to numData
numData=shrunkData.drop(cats, axis=1)
#getting the categorical data columns from shrunken data to be added to catData
catData=shrunkData.drop(nums, axis=1)
#removing the age label from temporary var
cats.remove(label)

In [ ]:
#Scaling the data for polynomicalFeatures(adding features)
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
#var to hold polynomial features
poly=PolynomialFeatures(degree=2)
#creating scaler
scaler=StandardScaler()
#makes polynomial features and scales them along with the original numerical data
transNumData=pd.DataFrame(scaler.fit_transform(poly.fit_transform(numData)), columns=poly.get_feature_names_out(input_features=None))
transNumData.shape

(154789, 595)

In [ ]:
#oneHot encoding the categorical data and label encoder for numerical data
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
#binary data
duos=["biometric_login_used", "two_factor_auth_enabled", "uses_premium_features", "has_children"]
#creating endocers
lencoder=LabelEncoder()
ohencoder=OneHotEncoder(sparse_output=False)
#var to hold output
catDataList=[]

#creating data frame for each column from duo array that is appended into catDataList
for duo in duos:
    catDataList.append(pd.DataFrame(lencoder.fit_transform(catData[duo]), columns=[duo]))
#Creating new features for all categorical data answers and removing the original feature. Added to the categoricalDataList
catDataList.append(pd.DataFrame(ohencoder.fit_transform(catData.drop(duos, axis=1)), columns=ohencoder.get_feature_names_out(input_features=None)))
#putting all of the dataFrames into one
transCatData=pd.concat(catDataList, axis=1)
transCatData.shape

(154789, 71)

###The toDrop array is holding all of the features who's threshold wasn't significant enough to use in training. It uses the correlation matric and the for loop 'for feature in unifiedData.columns:'

It takes forever to run, so once we ran it and found the output we use just one var to hold it so we don't waste time

In [ ]:
#Creating one var to hold all of the data
unifiedData=pd.concat([transCatData, transNumData, shrunkData[[label]]], axis=1)
#var holding to drop features
toDrop=['biometric_login_used', 'two_factor_auth_enabled', 'uses_premium_features', 'has_children', 'gender_Female', 'gender_Male', 'gender_Non-binary',
        'gender_Prefer not to say', 'country_Australia', 'country_Brazil', 'country_Canada', 'country_Germany', 'country_India', 'country_Japan',
        'country_Other', 'country_South Korea', 'country_United Kingdom', 'country_United States', 'urban_rural_Rural', 'urban_rural_Suburban',
        'urban_rural_Urban', 'income_level_High', 'income_level_Low', 'income_level_Lower-middle', 'income_level_Middle', 'income_level_Upper-middle',
        'employment_status_Freelancer', 'employment_status_Full-time employed', 'employment_status_Part-time', 'employment_status_Retired',
        'employment_status_Student', 'employment_status_Unemployed', 'education_level_Bachelor’s', 'education_level_High school',
        'education_level_Master’s', 'education_level_Other', 'education_level_PhD', 'education_level_Some college', 'relationship_status_Divorced',
        'relationship_status_In a relationship', 'relationship_status_Married', 'relationship_status_Single', 'relationship_status_Widowed',
        'smoking_Former', 'smoking_No', 'smoking_Yes', 'alcohol_frequency_Daily', 'alcohol_frequency_Never', 'alcohol_frequency_Rarely',
        'alcohol_frequency_Several times a week', 'alcohol_frequency_Weekly', 'content_type_preference_Live', 'content_type_preference_Mixed',
        'content_type_preference_Photos', 'content_type_preference_Reels', 'content_type_preference_Stories', 'content_type_preference_Videos',
        'preferred_content_theme_Art', 'preferred_content_theme_Fashion', 'preferred_content_theme_Fitness', 'preferred_content_theme_Food',
        'preferred_content_theme_Music', 'preferred_content_theme_Other', 'preferred_content_theme_Tech', 'preferred_content_theme_Travel',
        'privacy_setting_level_Friends only', 'privacy_setting_level_Private', 'privacy_setting_level_Public', 'subscription_status_Business',
        'subscription_status_Free', 'subscription_status_Premium', 'exercise_hours_per_week', 'sleep_hours_per_night', 'perceived_stress_score',
        'self_reported_happiness', 'daily_steps_count', 'weekly_work_hours', 'hobbies_count', 'social_events_per_month', 'books_read_per_year',
        'volunteer_hours_per_month', 'travel_frequency_per_year', 'notification_response_rate', 'account_creation_year', 'linked_accounts_count',
        'exercise_hours_per_week^2', 'exercise_hours_per_week sleep_hours_per_night', 'exercise_hours_per_week perceived_stress_score',
        'exercise_hours_per_week self_reported_happiness', 'exercise_hours_per_week daily_steps_count', 'exercise_hours_per_week weekly_work_hours',
        'exercise_hours_per_week hobbies_count', 'exercise_hours_per_week social_events_per_month', 'exercise_hours_per_week books_read_per_year',
        'exercise_hours_per_week volunteer_hours_per_month', 'exercise_hours_per_week travel_frequency_per_year',
        'exercise_hours_per_week notification_response_rate', 'exercise_hours_per_week account_creation_year',
        'exercise_hours_per_week linked_accounts_count', 'sleep_hours_per_night^2', 'sleep_hours_per_night perceived_stress_score',
        'sleep_hours_per_night self_reported_happiness', 'sleep_hours_per_night daily_steps_count', 'sleep_hours_per_night weekly_work_hours',
        'sleep_hours_per_night hobbies_count', 'sleep_hours_per_night social_events_per_month', 'sleep_hours_per_night books_read_per_year',
        'sleep_hours_per_night volunteer_hours_per_month', 'sleep_hours_per_night travel_frequency_per_year',
        'sleep_hours_per_night notification_response_rate', 'sleep_hours_per_night account_creation_year',
        'sleep_hours_per_night linked_accounts_count', 'perceived_stress_score^2', 'perceived_stress_score self_reported_happiness',
        'perceived_stress_score daily_steps_count', 'perceived_stress_score weekly_work_hours', 'perceived_stress_score hobbies_count',
        'perceived_stress_score social_events_per_month', 'perceived_stress_score books_read_per_year',
        'perceived_stress_score volunteer_hours_per_month', 'perceived_stress_score travel_frequency_per_year',
        'perceived_stress_score notification_response_rate', 'perceived_stress_score account_creation_year',
        'perceived_stress_score linked_accounts_count', 'self_reported_happiness^2', 'self_reported_happiness daily_steps_count',
        'self_reported_happiness weekly_work_hours', 'self_reported_happiness hobbies_count', 'self_reported_happiness social_events_per_month',
        'self_reported_happiness books_read_per_year', 'self_reported_happiness volunteer_hours_per_month',
        'self_reported_happiness travel_frequency_per_year', 'self_reported_happiness notification_response_rate',
        'self_reported_happiness account_creation_year', 'self_reported_happiness linked_accounts_count', 'daily_steps_count^2',
        'daily_steps_count weekly_work_hours', 'daily_steps_count hobbies_count', 'daily_steps_count social_events_per_month',
        'daily_steps_count books_read_per_year', 'daily_steps_count volunteer_hours_per_month', 'daily_steps_count travel_frequency_per_year',
        'daily_steps_count notification_response_rate', 'daily_steps_count account_creation_year', 'daily_steps_count linked_accounts_count',
        'weekly_work_hours^2', 'weekly_work_hours hobbies_count', 'weekly_work_hours social_events_per_month', 'weekly_work_hours books_read_per_year',
        'weekly_work_hours volunteer_hours_per_month', 'weekly_work_hours travel_frequency_per_year', 'weekly_work_hours notification_response_rate',
        'weekly_work_hours account_creation_year', 'weekly_work_hours linked_accounts_count', 'hobbies_count^2', 'hobbies_count social_events_per_month',
        'hobbies_count books_read_per_year', 'hobbies_count volunteer_hours_per_month', 'hobbies_count travel_frequency_per_year',
        'hobbies_count notification_response_rate', 'hobbies_count account_creation_year', 'hobbies_count linked_accounts_count',
        'social_events_per_month^2', 'social_events_per_month books_read_per_year', 'social_events_per_month volunteer_hours_per_month',
        'social_events_per_month travel_frequency_per_year', 'social_events_per_month notification_response_rate',
        'social_events_per_month account_creation_year', 'social_events_per_month linked_accounts_count', 'books_read_per_year^2',
        'books_read_per_year volunteer_hours_per_month', 'books_read_per_year travel_frequency_per_year',
        'books_read_per_year notification_response_rate', 'books_read_per_year account_creation_year', 'books_read_per_year linked_accounts_count',
        'volunteer_hours_per_month^2', 'volunteer_hours_per_month travel_frequency_per_year', 'volunteer_hours_per_month notification_response_rate',
        'volunteer_hours_per_month account_creation_year', 'volunteer_hours_per_month linked_accounts_count', 'travel_frequency_per_year^2',
        'travel_frequency_per_year notification_response_rate', 'travel_frequency_per_year account_creation_year',
        'travel_frequency_per_year linked_accounts_count', 'notification_response_rate^2', 'notification_response_rate account_creation_year',
        'notification_response_rate linked_accounts_count', 'account_creation_year^2', 'account_creation_year linked_accounts_count',
        'linked_accounts_count^2']

In [ ]:
#corrMat=unifiedData.corr()
#commented out due to run time issue, the output of this cell is the large array above
#threshold=.007
#toDrop=[]
#for feature in unifiedData.columns:
#    if abs(corrMat[label][feature]) < threshold:
#        toDrop.append(feature)
print(len(toDrop), toDrop)

190 ['biometric_login_used', 'two_factor_auth_enabled', 'uses_premium_features', 'has_children', 'gender_Female', 'gender_Male', 'gender_Non-binary', 'gender_Prefer not to say', 'country_Australia', 'country_Brazil', 'country_Canada', 'country_Germany', 'country_India', 'country_Japan', 'country_Other', 'country_South Korea', 'country_United Kingdom', 'country_United States', 'urban_rural_Rural', 'urban_rural_Suburban', 'urban_rural_Urban', 'income_level_High', 'income_level_Low', 'income_level_Lower-middle', 'income_level_Middle', 'income_level_Upper-middle', 'employment_status_Freelancer', 'employment_status_Full-time employed', 'employment_status_Part-time', 'employment_status_Retired', 'employment_status_Student', 'employment_status_Unemployed', 'education_level_Bachelor’s', 'education_level_High school', 'education_level_Master’s', 'education_level_Other', 'education_level_PhD', 'education_level_Some college', 'relationship_status_Divorced', 'relationship_status_In a relationship'

In [ ]:
print("unified data shape: ",unifiedData.shape)
#Dropping the data features from the unfiied data into a new var named pruned data
prunedData=unifiedData.drop(toDrop, axis=1)
print('features to be dropped: ', len(toDrop))
print('prunedData shape:',prunedData.shape)

unified data shape:  (154789, 667)
features to be dropped:  190
prunedData shape: (154789, 477)


In [ ]:
trainData, testData=splitData(prunedData, firstRatio=.95)


XtrainFull=trainData.drop(label, axis=1)
ytrainFull=trainData[label].copy()

#creation of validation set for the ANN Regression sequential API
Xtrain, Xval = splitData(XtrainFull, firstRatio =.75)
ytrain, yval = splitData(ytrainFull, firstRatio =.75)



Xtest=testData.drop(label, axis=1).to_numpy()
ytest=testData[label].copy().to_numpy()
print("Train: ", Xtrain.shape, ytrain.shape, "\nValidatino: ",  Xval.shape, yval.shape, "\nTest: ", Xtest.shape, ytest.shape)

Train:  (110286, 476) (110286,) 
Validatino:  (36763, 476) (36763,) 
Test:  (7740, 476) (7740,)


#SGD Regression

In [ ]:
from sklearn.linear_model import SGDRegressor
sgdReg=SGDRegressor()
sgdReg.fit(XtrainFull, ytrainFull)
print(sgdReg.score(XtrainFull, ytrainFull), sgdReg.score(Xtest, ytest))

0.6277704989419852 0.6235901233890149


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but SGDRegressor was fitted with feature names
  warnings.warn(


#ANN regression with sequential API

In [ ]:
import tensorflow as tf

tf.random.set_seed(42)

XtrainNp = Xtrain.to_numpy(dtype=np.float32)
XvalNp = Xval.to_numpy(dtype=np.float32)

norm_layer = tf.keras.layers.Normalization(axis = 1)
norm_layer.adapt(XtrainNp)

model = tf.keras.Sequential([
    norm_layer,
    tf.keras.layers.Dense(250, activation="relu"),
    tf.keras.layers.Dense(250, activation="relu"),
    tf.keras.layers.Dense(250, activation="relu"),
    tf.keras.layers.Dense(1)#no activation and regression only has one neuron in output layer
])

optmzr = tf.keras.optimizers.Adam(learning_rate=1e-3)
model.compile(loss="mse", optimizer=optmzr, metrics=["mse"])

from keras.callbacks import EarlyStopping

epchs = 30

monitor_loss = EarlyStopping(monitor='mse',patience=3,
                             min_delta = 0.002)


history = model.fit(Xtrain, ytrain, epochs=epchs,
                    validation_data=(Xval, yval))



Epoch 1/30
3447/3447 ━━━━━━━━━━━━━━━━━━━━ 26s 7ms/step - loss: 302.9501 - mse: 302.9501 - val_loss: 253.6165 - val_mse: 253.6165
Epoch 2/30
3447/3447 ━━━━━━━━━━━━━━━━━━━━ 16s 5ms/step - loss: 242.6467 - mse: 242.6467 - val_loss: 246.8241 - val_mse: 246.8241
Epoch 3/30
3447/3447 ━━━━━━━━━━━━━━━━━━━━ 16s 5ms/step - loss: 240.8752 - mse: 240.8752 - val_loss: 240.7546 - val_mse: 240.7546
Epoch 4/30
3447/3447 ━━━━━━━━━━━━━━━━━━━━ 16s 5ms/step - loss: 239.5654 - mse: 239.5654 - val_loss: 239.4194 - val_mse: 239.4194
Epoch 5/30
3447/3447 ━━━━━━━━━━━━━━━━━━━━ 20s 5ms/step - loss: 238.7164 - mse: 238.7164 - val_loss: 238.8536 - val_mse: 238.8536
Epoch 6/30
3447/3447 ━━━━━━━━━━━━━━━━━━━━ 16s 5ms/step - loss: 237.9785 - mse: 237.9785 - val_loss: 238.4494 - val_mse: 238.4494
Epoch 7/30
3447/3447 ━━━━━━━━━━━━━━━━━━━━ 16s 5ms/step - loss: 237.4403 - mse: 237.4403 - val_loss: 238.1257 - val_mse: 238.1257
Epoch 8/30
3447/3447 ━━━━━━━━━━━━━━━━━━━━ 16s 5ms/step - loss: 237.0069 - mse: 237.0069 - val_los

In [ ]:
ranEpochs = monitor_loss.stopped_epoch + 1
maxLoss =  max(history.history['loss'])
minLoss = min(history.history['loss'])

maxMse =  max(history.history['mse'])
minMse = min(history.history['mse'])

print("MaxLoss:", maxLoss,
      "\nMinLiss:", minLoss,
      "\nMaxMse:", maxMse,
      "\nMinMse:", maxMse
      )



MaxLoss: 254.8222198486328 
MinLiss: 229.60011291503906 
MaxMse: 254.8222198486328 
MinMse: 254.8222198486328


#Output of ANN regression

>MaxLoss: 254.8222198486328

>MinLiss: 229.60011291503906

>MaxMse: 254.8222198486328

>MinMse: 254.8222198486328

Epoch 1/30
3447/3447 ━━━━━━━━━━━━━━━━━━━━ 19s 5ms/step - loss: 291.4637 - mse: 291.4637 - val_loss: 256.7123 - val_mse: 256.7123

Epoch 2/30
3447/3447 ━━━━━━━━━━━━━━━━━━━━ 20s 5ms/step - loss: 243.1612 - mse: 243.1612 - val_loss: 245.3878 - val_mse: 245.3878

Epoch 3/30
3447/3447 ━━━━━━━━━━━━━━━━━━━━ 17s 5ms/step - loss: 240.9834 - mse: 240.9834 - val_loss: 241.2258 - val_mse: 241.2258

Epoch 4/30
3447/3447 ━━━━━━━━━━━━━━━━━━━━ 22s 5ms/step - loss: 239.6391 - mse: 239.6391 - val_loss: 239.7052 - val_mse: 239.7052

Epoch 5/30
3447/3447 ━━━━━━━━━━━━━━━━━━━━ 17s 5ms/step - loss: 238.7007 - mse: 238.7007 - val_loss: 239.0096 - val_mse: 239.0096

Epoch 6/30
3447/3447 ━━━━━━━━━━━━━━━━━━━━ 20s 5ms/step - loss: 237.9782 - mse: 237.9782 - val_loss: 238.5977 - val_mse: 238.5977

Epoch 7/30
3447/3447 ━━━━━━━━━━━━━━━━━━━━ 16s 5ms/step - loss: 237.4408 - mse: 237.4408 - val_loss: 238.1069 - val_mse: 238.1069

Epoch 8/30
3447/3447 ━━━━━━━━━━━━━━━━━━━━ 22s 5ms/step - loss: 236.9594 - mse: 236.9594 - val_loss: 238.1402 - val_mse: 238.1402

Epoch 9/30
3447/3447 ━━━━━━━━━━━━━━━━━━━━ 16s 5ms/step - loss: 236.6349 - mse: 236.6349 - val_loss: 237.9799 - val_mse: 237.9799

Epoch 10/30
3447/3447 ━━━━━━━━━━━━━━━━━━━━ 16s 5ms/step - loss: 236.3022 - mse: 236.3022 - val_loss: 238.2748 - val_mse: 238.2748

Epoch 11/30
3447/3447 ━━━━━━━━━━━━━━━━━━━━ 20s 5ms/step - loss: 235.9938 - mse: 235.9938 - val_loss: 238.2979 - val_mse: 238.2979

Epoch 12/30
3447/3447 ━━━━━━━━━━━━━━━━━━━━ 16s 5ms/step - loss: 235.7582 - mse: 235.7582 - val_loss: 238.3426 - val_mse: 238.3426

Epoch 13/30
3447/3447 ━━━━━━━━━━━━━━━━━━━━ 17s 5ms/step - loss: 235.4727 - mse: 235.4727 -
val_loss: 238.8425 - val_mse: 238.8425

Epoch 14/30
3447/3447 ━━━━━━━━━━━━━━━━━━━━ 16s 5ms/step - loss: 235.2676 - mse: 235.2676 - val_loss: 239.6388 - val_mse: 239.6388

Epoch 15/30
3447/3447 ━━━━━━━━━━━━━━━━━━━━ 17s 5ms/step - loss: 235.0443 - mse: 235.0443 - val_loss: 239.7409 - val_mse: 239.7409

Epoch 16/30
3447/3447 ━━━━━━━━━━━━━━━━━━━━ 16s 5ms/step - loss: 234.7270 - mse: 234.7270 - val_loss: 240.5260 - val_mse: 240.5260

Epoch 17/30
3447/3447 ━━━━━━━━━━━━━━━━━━━━ 16s 5ms/step - loss: 234.4542 - mse: 234.4542 - val_loss: 241.1291 - val_mse: 241.1291

Epoch 18/30
3447/3447 ━━━━━━━━━━━━━━━━━━━━ 22s 5ms/step - loss: 234.1645 - mse: 234.1645 - val_loss: 240.7980 - val_mse: 240.7980

Epoch 19/30
3447/3447 ━━━━━━━━━━━━━━━━━━━━ 20s 6ms/step - loss: 233.7684 - mse: 233.7684 - val_loss: 242.8490 - val_mse: 242.8490

Epoch 20/30
3447/3447 ━━━━━━━━━━━━━━━━━━━━ 23s 7ms/step - loss: 233.3878 - mse: 233.3878 - val_loss: 243.6380 - val_mse: 243.6380

Epoch 21/30
3447/3447 ━━━━━━━━━━━━━━━━━━━━ 24s 7ms/step - loss: 233.0397 - mse: 233.0397 - val_loss: 241.4501 - val_mse: 241.4501

Epoch 22/30
3447/3447 ━━━━━━━━━━━━━━━━━━━━ 19s 5ms/step - loss: 232.7632 - mse: 232.7632 - val_loss: 244.3803 - val_mse: 244.3803

Epoch 23/30
3447/3447 ━━━━━━━━━━━━━━━━━━━━ 19s 6ms/step - loss: 232.4910 - mse: 232.4910 - val_loss: 243.9738 - val_mse: 243.9738

Epoch 24/30
3447/3447 ━━━━━━━━━━━━━━━━━━━━ 20s 5ms/step - loss: 232.0755 - mse: 232.0755 - val_loss: 244.4420 - val_mse: 244.4420

Epoch 25/30
3447/3447 ━━━━━━━━━━━━━━━━━━━━ 18s 5ms/step - loss: 231.5385 - mse: 231.5385 - val_loss: 245.3591 - val_mse: 245.3591

Epoch 26/30
3447/3447 ━━━━━━━━━━━━━━━━━━━━ 21s 6ms/step - loss: 231.2116 - mse: 231.2116 - val_loss: 245.3014 - val_mse: 245.3014

Epoch 27/30
3447/3447 ━━━━━━━━━━━━━━━━━━━━ 17s 5ms/step - loss: 230.7205 - mse: 230.7205 - val_loss: 245.0269 - val_mse: 245.0269

Epoch 28/30
3447/3447 ━━━━━━━━━━━━━━━━━━━━ 17s 5ms/step - loss: 230.1880 - mse: 230.1880 - val_loss: 246.3807 - val_mse: 246.3807

Epoch 29/30
3447/3447 ━━━━━━━━━━━━━━━━━━━━ 18s 5ms/step - loss: 229.7514 - mse: 229.7514 - val_loss: 246.6612 - val_mse: 246.6612

Epoch 30/30
3447/3447 ━━━━━━━━━━━━━━━━━━━━ 17s 5ms/step - loss: 229.3591 - mse: 229.3591 - val_loss: 247.0002 - val_mse: 247.0002